In [ ]:

import torch
import torch.nn as nn


In [ ]:
class CoreferenceModel(nn.Module):
    def __init__(self, mention_dim=2304, hidden_dim=150):
        super().__init__()

        # Pair representation:
        # [mention_i ; mention_j ; mention_i * mention_j]
        # CORRECT FORMULA NOW :D
        # 2304 * 3 = 6912 dimensions

        self.mlp = nn.Sequential(
            ##LinearLayer
            nn.Linear(mention_dim * 3, hidden_dim), ## 6912 → 150
            nn.ReLU(),
            nn.Linear(hidden_dim, 1) # # 150 → 1
        )

    def forward(self, mention_embeddings):
        """
        Input:
            mention_embeddings: (N, 2304)

        Output:
            logits: (num_pairs,)
            pairs: list of (i, j)
        """

        N = mention_embeddings.shape[0] #number of mentions from (N, 2304)

        pairs = [
            (i, j)
            for i in range(N)
            for j in range(i + 1, N)
        ]

        if not pairs: #if list is empty or no pairs exist, return empty tensor and empty pair list
            return mention_embeddings.new_empty((0,)), pairs

        # Gather embeddings for every mention pair, separates the first and second elements:
        i_idx, j_idx = zip(*pairs)

        m_i = mention_embeddings[list(i_idx)]
        m_j = mention_embeddings[list(j_idx)]

        # (num_pairs, 6912) architecture's pair representation,[mention_i ; mention_j ; mention_i * mention_j] 
        pair_vectors = torch.cat(
            [m_i, m_j, m_i * m_j],
            dim=-1
        )

        # (num_pairs,)
        logits = self.mlp(pair_vectors).squeeze(-1) #drop the extra dimension so output is (num_pairs,)

        return logits, pairs


    ##For the threshold, can be changed, i just kept it as 0.5 for now
    @torch.no_grad()
    def predict(self, mention_embeddings, threshold=0.5 ):
        """
        Returns predicted entity clusters.
        """
        
        #SHould be  no dropout or batch normalization but ill still keep it in 
        self.eval()

        logits, pairs = self.forward(mention_embeddings)
        probabilities = torch.sigmoid(logits)

        N = mention_embeddings.shape[0]

        # Union-Find clustering
        parent = list(range(N))


        #determines which cluster  x belongs to by following parent 
        #links until it reaches the representative of the cluster.
        #basically shortens shortens the path to the representative
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        
        #combines the clusters containing mentions a and b, combining those with high probs into same cluster
        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb: #at this point a and b already passed the threshold so they are now linked
                parent[rb] = ra

        for (i, j), prob in zip(pairs, probabilities.tolist()):
            if prob >= threshold:
                union(i, j)

        clusters = {}
        
        for i in range(N):
            root = find(i)
            if root not in clusters:
                clusters[root] = []
            
            clusters[root].append(i)

        return list(clusters.values())
